# Induction Evals
We evaluate that the induction case is strong enough to drive a wedge between LLM performance in this notebook.

In [11]:
"""The following generates the Quiz all our models will be evaluated on."""
import itertools
import string
import numpy as np

import logging

from ordered_set import OrderedSet
from dotenv import load_dotenv

logging.basicConfig(
	level=logging.DEBUG
)
load_dotenv("keys.env", verbose=True)

from smolbench.induction.chromatic import (
	ChromaticIntervalsConfig,
	Prompter,
	get_random_exclusive_quiz,
	anneal_intervals
)
from smolbench.evals import (
	Marks
)
from smolbench.evals.openrouter import (
	evaluate
)

template = string.Template(
	"You are a Boolean classifier.\n"
	"\n"
	"Task: determine whether the statement in the Question is logically "
	"possible given the Context.\n"
	"\n"
	"Output format:\n"
	"Return exactly one of these two strings and nothing else:\n"
	"True\n"
	"False\n"
	"\n"
	"Do not output any explanation, punctuation, quotes, labels, code fences, "
	"or extra whitespace."
	"Stop immediately after writing True or False."
	"\n"
	"Context:\n"
	"There is a ceremonial role called the $role, whose job it is to"
	" head the $parade parade. No one else besides the $role is able to head"
	" the $parade parade. The following lists the people who were $role and"
	" the years they were $role:\n"
	"$positive_info\n"
	"\n"
	"Question:\n"
	"Between the years $start and $end, exclusive of the end, could $color"
	" have headed the $parade parade every year?"
)

def query_gen(
	labels_to_intervals: Dict[Color, Intervals],
	interval_to_label: Dict[Intervals, Color],
	seed: int,
) -> Dict[str, str]:
	"""Generates a series of queries"""
	rng: np.random.Generator = np.random.default_rng(seed)
	# Finds max interval.
	n: int = max(interval[1] for interval in interval_to_label)
	for color, intervals in labels_to_intervals.items():
		# Generates a series of true items.
		for start, end in intervals:
			start, end = np.sort(
				rng.choice(range(start, end + 1), size=2, replace=False)
			)
			yield {"color": color, "start": start, "end": end}, True
		# Generates a false proposition.
		invalid_range: Intervals = anneal_intervals(
			itertools.chain(
				(OrderedSet(interval_to_label.keys()) - OrderedSet(itertools.chain(*intervals)))
			)
		)
		for start, end in invalid_range:
			start = rng.choice(range(start, end))
			# Binom with p = intervals / n capped at end for a similar-ish
			# distr. to positive accounts.
			end = min(
				end,
				start
				+ rng.binomial(
					end - start + 1,
					np.mean([len(interval) for interval in interval_to_label]) / n,
				)
				+ 1,
			)
			yield {"color": color, "start": start, "end": end}, False


SEED: int = 1776
intens_quiz, extens_quiz = get_random_exclusive_quiz(
	ChromaticIntervalsConfig(
            n=250,
            intervals=250 // 4,
            colors=45,
            seed=SEED,
        ),
	Prompter(
		template,
		{
			"role": "Twislax",
			"parade": "Gildane",
		},
		query_gen,
	),
)

## Prompt Validation

In [2]:
print(intens_quiz[0].prompt)

You are a Boolean classifier.

Task: determine whether the statement in the Question is logically possible given the Context.

Output format:
Return exactly one of these two strings and nothing else:
True
False

Do not output any explanation, punctuation, quotes, labels, code fences, or extra whitespace.Stop immediately after writing True or False.
Context:
There is a ceremonial role called the Twislax, whose job it is to head the Gildane parade. No one else besides the Twislax is able to head the Gildane parade. The following lists the people who were Twislax and the years they were Twislax:
qo was Twislax on 24 to 34.
Lw was Twislax on 108 to 120 and 137 to 139.
Gv was Twislax on 35 to 36 and 134 to 135.
Wd was Twislax on 63 to 63.
UE was Twislax on 186 to 190.
ld was Twislax on 232 to 233.
Gt was Twislax on 88 to 90.
nR was Twislax on 38 to 41 and 201 to 204.
Ef was Twislax on 0 to 10, 180 to 181, and 225 to 226.
wJ was Twislax on 61 to 62, 149 to 160, and 191 to 200.
gB was Twislax

In [3]:
print(extens_quiz[0].prompt)

You are a Boolean classifier.

Task: determine whether the statement in the Question is logically possible given the Context.

Output format:
Return exactly one of these two strings and nothing else:
True
False

Do not output any explanation, punctuation, quotes, labels, code fences, or extra whitespace.Stop immediately after writing True or False.
Context:
There is a ceremonial role called the Twislax, whose job it is to head the Gildane parade. No one else besides the Twislax is able to head the Gildane parade. The following lists the people who were Twislax and the years they were Twislax:
qo was Twislax on 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, and 34.
Lw was Twislax on 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 137, 138, and 139.
Gv was Twislax on 35, 36, 134, and 135.
Wd was Twislax on 63.
UE was Twislax on 186, 187, 188, 189, and 190.
ld was Twislax on 232 and 233.
Gt was Twislax on 88, 89, and 90.
nR was Twislax on 38, 39, 40, 41, 201, 202, 203, and 204.


# Decoder-Only Model
This section tests classical decoder-only models.

In [ ]:
decode_intens_eval: Marks = evaluate(intens_quiz, "mistralai/mistral-7b-instruct-v0.1", SEED)

In [ ]:
decode_extens_eval: Marks = evaluate(extens_quiz, "mistralai/mistral-7b-instruct-v0.1", SEED)

In [6]:
print(decode_intens_eval.correct, decode_intens_eval.incorrect, decode_intens_eval.invalid)
print(decode_extens_eval.correct, decode_extens_eval.incorrect, decode_extens_eval.invalid)

36 58 13
54 42 11


## CoT Model
This section tests a CoT model.

In [ ]:
cot_intens_eval: Marks = evaluate(intens_quiz, "deepseek/deepseek-r1-distill-qwen-32b", SEED)

In [ ]:
cot_extens_eval: Marks = evaluate(extens_quiz, "deepseek/deepseek-r1-distill-qwen-32b", SEED)

In [9]:
print(cot_intens_eval.correct, cot_intens_eval.incorrect, cot_intens_eval.invalid)
print(cot_extens_eval.correct, cot_extens_eval.incorrect, cot_extens_eval.invalid)

105 1 1
104 2 1


## MoE Model
This section tests an MoE model.

In [ ]:
moe_intens_eval: Marks = evaluate(intens_quiz, "liquid/lfm2-8b-a1b", SEED)

DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): openrouter.ai:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): openrouter.ai:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): openrouter.ai:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): openrouter.ai:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): openrouter.ai:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): openrouter.ai:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): openrouter.ai:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): openrouter.ai:443
DEBUG:urllib3.connectionpool:https://openrouter.ai:443 "POST /api/v1/chat/completions HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://openrouter.ai:443 "POST /api/v1/chat/completions HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://openrouter.ai:443 "POST /api/v1/chat/completions HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:

KeyError: 'choices'

DEBUG:urllib3.connectionpool:https://openrouter.ai:443 "POST /api/v1/chat/completions HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://openrouter.ai:443 "POST /api/v1/chat/completions HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://openrouter.ai:443 "POST /api/v1/chat/completions HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://openrouter.ai:443 "POST /api/v1/chat/completions HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://openrouter.ai:443 "POST /api/v1/chat/completions HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://openrouter.ai:443 "POST /api/v1/chat/completions HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): openrouter.ai:443
DEBUG:root:{'error': {'message': 'Provider returned error', 'code': 504}}
DEBUG:root:{'error': {'message': 'Provider returned error', 'code': 504}}
DEBUG:root:{'error': {'message': 'Provider returned error', 'code': 504}}
DEBUG:root:{'error': {'message': 'Provider returned error', 'code'

In [ ]:
moe_extens_eval: Marks = evaluate(extens_quiz, "liquid/lfm2-8b-a1b", SEED)

In [ ]:
print(moe_intens_eval.correct, moe_intens_eval.incorrect, moe_intens_eval.invalid)
print(moe_extens_eval.correct, moe_extens_eval.incorrect, moe_extens_eval.invalid)

# HRM Model
This section tests an HRM model.

In [ ]:
hrm_intens_eval: Marks = evaluate(intens_quiz, "perplexity/pplx-embed-v1-4b", SEED)

In [ ]:
hrm_extens_eval: Marks = evaluate(extens_quiz, "perplexity/pplx-embed-v1-4b", SEED)

In [ ]:
print(hrm_intens_eval.correct, hrm_intens_eval.incorrect, hrm_intens_eval.invalid)
print(hrm_extens_eval.correct, hrm_extens_eval.incorrect, hrm_extens_eval.invalid)